# 3. Routing

*Using Microsoft Semantic Kernel (Agent Framework)*

Demonstrates how to route requests to different specialized agents based on the input. This pattern allows you to create a system where different types of queries are handled by agents optimized for specific tasks.

This notebook uses Semantic Kernel to implement routing between specialized agents.

In [ ]:
import os
from typing import Annotated
from dotenv import load_dotenv
import semantic_kernel as sk
from semantic_kernel.connectors.ai.open_ai import AzureChatCompletion
from semantic_kernel.contents import ChatHistory
from semantic_kernel.functions import kernel_function
from semantic_kernel.connectors.ai.open_ai.prompt_execution_settings.azure_chat_prompt_execution_settings import (
    AzureChatPromptExecutionSettings,
)
from semantic_kernel.connectors.ai.function_choice_behavior import FunctionChoiceBehavior

Initialize the kernel

In [ ]:
load_dotenv()

kernel = sk.Kernel()
service_id = "chat-gpt"
kernel.add_service(
    AzureChatCompletion(
        service_id=service_id,
        deployment_name="gpt-4o-mini",
        endpoint=os.getenv("AZURE_OPENAI_ENDPOINT"),
        api_key=os.getenv("AZURE_OPENAI_API_KEY"),
    )
)

Define specialized plugins for routing

In [ ]:
class WeatherPlugin:
    @kernel_function(
        name="get_weather",
        description="Get weather information for a city"
    )
    def get_weather(self, location: Annotated[str, "City name"]) -> str:
        weather_data = {
            "New York": "Sunny, 25°C",
            "Los Angeles": "Cloudy, 22°C",
            "Chicago": "Rainy, 18°C",
        }
        return weather_data.get(location, "Weather data not available.")

class TravelPlugin:
    @kernel_function(
        name="get_destination",
        description="Get travel destination recommendations"
    )
    def get_destination(self, interest: Annotated[str, "User interest or preference"]) -> str:
        destinations = {
            "beach": "Maldives - Beautiful beaches and crystal clear water",
            "mountain": "Swiss Alps - Stunning mountain views and hiking",
            "city": "Tokyo - Vibrant city life and culture",
        }
        for key, dest in destinations.items():
            if key in interest.lower():
                return dest
        return "Paris - A versatile destination for all interests"

# Add plugins to kernel
kernel.add_plugin(WeatherPlugin(), plugin_name="weather")
kernel.add_plugin(TravelPlugin(), plugin_name="travel")

Create router function that directs queries to appropriate agents

In [ ]:
async def route_query(query: str) -> str:
    """Route the query to the appropriate specialized agent."""
    chat_history = ChatHistory()
    chat_history.add_system_message(
        "You are a helpful assistant that can provide weather information and travel recommendations. "
        "Use the appropriate tool based on the user's question."
    )
    chat_history.add_user_message(query)
    
    execution_settings = AzureChatPromptExecutionSettings(
        service_id=service_id,
        function_choice_behavior=FunctionChoiceBehavior.Auto(),
    )
    
    chat_service = kernel.get_service(service_id)
    response = await chat_service.get_chat_message_content(
        chat_history=chat_history,
        settings=execution_settings,
        kernel=kernel,
    )
    
    return str(response)

Test routing with different queries

In [ ]:
# Weather query - should route to weather agent
result = await route_query("What's the weather in Chicago?")
print("Weather Query Result:")
print(result)
print()

In [ ]:
# Travel query - should route to travel agent
result = await route_query("I want to visit a beach destination")
print("Travel Query Result:")
print(result)

**Note:** Semantic Kernel's automatic function calling handles routing by selecting the appropriate tool based on the query context. This provides similar functionality to LangGraph's routing patterns.